In [2]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [3]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [4]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [6]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [7]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [8]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [9]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [10]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-16 07:44:22 | people | execute | Started
2025-03-16 07:44:22 | people | execute | Started
2025-03-16 07:44:22 | people | load | Started
2025-03-16 07:44:27 | people | load | Completed in 0.08 min
2025-03-16 07:44:27 | people | transform | Started
2025-03-16 07:44:28 | people | transform | Completed in 0.0 min
2025-03-16 07:44:28 | people | write | Started
2025-03-16 07:44:52 | people | write | Completed in 0.4 min
2025-03-16 07:44:52 | people | execute | Completed in 0.48 min
2025-03-16 07:44:52 | planets | execute | Started
2025-03-16 07:44:52 | planets | load | Started
2025-03-16 07:44:54 | planets | load | Completed in 0.03 min
2025-03-16 07:44:54 | planets | transform | Started
2025-03-16 07:44:55 | planets | transform | Completed in 0.0 min
2025-03-16 07:44:55 | planets | write | Started
2025-03-16 07:45:10 | planets | write | Completed in 0.25 min
2025-03-16 07:45:10 | planets | execute | Completed in 0.3 min
2025-03-16 07:45:10 | people | execute | Completed in 0.78 min


In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-03-16 07:44:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:44:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-03

In [12]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [13]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [14]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [15]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df


silver_instance = StarWarsSilver(
    spark, catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [16]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

2025-03-16 07:45:14 | people | execute | Started
2025-03-16 07:45:14 | people | execute | Started
2025-03-16 07:45:14 | people | load | Started
2025-03-16 07:45:14 | people | load | Completed in 0.0 min
2025-03-16 07:45:14 | people | transform | Started
2025-03-16 07:45:14 | people | transform | Completed in 0.0 min
2025-03-16 07:45:14 | people | write | Started
2025-03-16 07:45:17 | people | write | Completed in 0.03 min
2025-03-16 07:45:17 | people | execute | Completed in 0.03 min
2025-03-16 07:45:17 | planets | execute | Started
2025-03-16 07:45:17 | planets | load | Started
2025-03-16 07:45:17 | planets | load | Completed in 0.0 min
2025-03-16 07:45:17 | planets | transform | Started
2025-03-16 07:45:17 | planets | transform | Completed in 0.0 min
2025-03-16 07:45:17 | planets | write | Started
2025-03-16 07:45:18 | planets | write | Completed in 0.02 min
2025-03-16 07:45:18 | planets | execute | Completed in 0.02 min
2025-03-16 07:45:18 | people | execute | Completed in 0.07 min


In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+--------------------------+---------------------+---+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 18
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-03-16 07:45:

# 6 Clean Up

In [19]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.stop()